# Обучение головы предсказания статистик (39 статов)

Эмбеддинги берём **после блока трансформера** (выход энкодера). Обученная модель энкодера: `model_trained/embed_128`. Обучаем только голову `StatsPredictionHead`, энкодер заморожен.

In [1]:
import sys
from pathlib import Path

ROOT = Path(".").resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pickle
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from safetensors.torch import load_file as load_safetensors

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print("Device:", device)

Device: cuda


## 1. Пути и данные

In [2]:
ENCODER_CKPT_DIR = ROOT / "model_trained" / "embed_128"
ENCODER_CKPT_DIR_64 = ROOT / "model_trained" / "embed_64"
METADATA_DIR = ROOT / "notebooks" / "train_sample_processed" / "metadata"
PROCESSED_CSV = ROOT / "notebooks" / "train_sample_processed" / "processed.csv"
OUTPUT_DIR = ROOT / "notebooks" / "stats_head_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "embed_64").mkdir(parents=True, exist_ok=True)
HEAD_PATH_64 = OUTPUT_DIR / "embed_64" / "stats_head.pt"

assert ENCODER_CKPT_DIR.exists(), f"Не найден чекпоинт энкодера: {ENCODER_CKPT_DIR}"
assert ENCODER_CKPT_DIR_64.exists(), f"Не найден чекпоинт энкодера 64: {ENCODER_CKPT_DIR_64}"
assert METADATA_DIR.exists(), f"Не найден metadata: {METADATA_DIR}"
assert PROCESSED_CSV.exists(), f"Не найден processed CSV: {PROCESSED_CSV}"

In [3]:
import pandas as pd

def load_vocab(metadata_dir: Path):
    with open(metadata_dir / "player_name2id.pickle", "rb") as f:
        player_name2id = pickle.load(f)
    with open(metadata_dir / "team_name2id.pickle", "rb") as f:
        team_name2id = pickle.load(f)
    n_players = len(player_name2id)
    n_teams = len(team_name2id)
    return {
        "player_name2id": player_name2id,
        "team_name2id": team_name2id,
        "player_pad_token_id": n_players + 1,
        "team_pad_token_id": n_teams,
    }

vocab = load_vocab(METADATA_DIR)
df = pd.read_csv(PROCESSED_CSV)
print("Матчей:", df["match_id"].nunique(), "| Игроков в словаре:", len(vocab["player_name2id"]))

Матчей: 2923 | Игроков в словаре: 6393


## 2. Датасет и коллатор

In [4]:
from data.dataset import MatchDatasetMPP
from torch.utils.data import default_collate

def collate_stats(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        raise ValueError("All items in batch are None")
    return default_collate(batch)

max_seq_length = 36
ds_full = MatchDatasetMPP(
    df,
    player_name2id=vocab["player_name2id"],
    team_name2id=vocab["team_name2id"],
    max_seq_length=max_seq_length,
    player_pad_token_id=vocab["player_pad_token_id"],
    team_pad_token_id=vocab["team_pad_token_id"],
    position_pad_token_id=25,
)

n = len(ds_full)
n_val = max(1, int(n * 0.1))
n_train = n - n_val
train_ds = Subset(ds_full, range(0, n_train))
eval_ds = Subset(ds_full, range(n_train, n))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_stats, num_workers=0)
eval_loader = DataLoader(eval_ds, batch_size=32, shuffle=False, collate_fn=collate_stats, num_workers=0)
print("Train батчей:", len(train_loader), "| Eval батчей:", len(eval_loader))

Train батчей: 83 | Eval батчей: 10


## 3. Модель: энкодер (заморожен) + StatsPredictionHead

In [5]:
from models.encoder import PlayerEncoder
from models.heads import StatsPredictionHead
from data.preprocessing import FORM_STATS_SIZE

embed_size = 128
players_vocab_size = vocab["player_pad_token_id"]
teams_vocab_size = vocab["team_pad_token_id"] + 1

encoder = PlayerEncoder(
    embed_size=embed_size,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=FORM_STATS_SIZE,
    players_vocab_size=players_vocab_size,
    teams_vocab_size=teams_vocab_size,
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)

state = load_safetensors(ENCODER_CKPT_DIR / "model.safetensors")
encoder_state = {k.replace("encoder.", ""): v for k, v in state.items() if k.startswith("encoder.")}
encoder.load_state_dict(encoder_state, strict=True)
for p in encoder.parameters():
    p.requires_grad = False

head = StatsPredictionHead(embed_size=embed_size, num_stats=FORM_STATS_SIZE, hidden_dim=256)

encoder = encoder.to(device)
head = head.to(device)
print("Параметров головы:", sum(p.numel() for p in head.parameters()))

Параметров головы: 43047


## 4. Обучение

In [6]:
from training.stats_trainer import StatsHeadTrainer

trainer = StatsHeadTrainer(
    encoder=encoder,
    head=head,
    train_loader=train_loader,
    eval_loader=eval_loader,
    output_dir=str(OUTPUT_DIR),
    num_epochs=20,
    lr=1e-3,
    weight_decay=0.01,
    device=device,
    logging_steps=1,
    save_best=True,
)
trainer.train()

Epoch 1/20  Train MSE: 839.943736  Eval MSE: 374.143475
Epoch 2/20  Train MSE: 324.849434  Eval MSE: 308.160434
Epoch 3/20  Train MSE: 276.239966  Eval MSE: 276.805550
Epoch 4/20  Train MSE: 257.812642  Eval MSE: 264.495300
Epoch 5/20  Train MSE: 249.768053  Eval MSE: 253.101764
Epoch 6/20  Train MSE: 242.911207  Eval MSE: 248.164682
Epoch 7/20  Train MSE: 240.016530  Eval MSE: 253.932408
Epoch 8/20  Train MSE: 236.514970  Eval MSE: 244.232437
Epoch 9/20  Train MSE: 234.039634  Eval MSE: 248.922282
Epoch 10/20  Train MSE: 229.441773  Eval MSE: 241.425276
Epoch 11/20  Train MSE: 228.499472  Eval MSE: 241.176405
Epoch 12/20  Train MSE: 226.383524  Eval MSE: 239.185339
Epoch 13/20  Train MSE: 224.452483  Eval MSE: 245.157921
Epoch 14/20  Train MSE: 224.440637  Eval MSE: 243.435405
Epoch 15/20  Train MSE: 219.536853  Eval MSE: 238.190749
Epoch 16/20  Train MSE: 217.346676  Eval MSE: 237.911119
Epoch 17/20  Train MSE: 216.497554  Eval MSE: 233.763606
Epoch 18/20  Train MSE: 214.594255  Eval

233.76360626220702

In [7]:
# Обучение выполнено в ячейке выше через StatsHeadTrainer

## 4b. Обучение головы embed_64

Энкодер из `model_trained/embed_64` (64-мерные эмбеддинги). Обучаем только голову, энкодер заморожен; чекпоинт сохраняется в `stats_head_output/embed_64/stats_head.pt`.

In [10]:
encoder_64 = PlayerEncoder(
    embed_size=64,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=FORM_STATS_SIZE,
    players_vocab_size=players_vocab_size,
    teams_vocab_size=teams_vocab_size,
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
state_64 = load_safetensors(ENCODER_CKPT_DIR_64 / "model.safetensors")
enc_state_64 = {k.replace("encoder.", ""): v for k, v in state_64.items() if k.startswith("encoder.")}
encoder_64.load_state_dict(enc_state_64, strict=True)
for p in encoder_64.parameters():
    p.requires_grad = False

head_64 = StatsPredictionHead(embed_size=64, num_stats=FORM_STATS_SIZE, hidden_dim=256)
encoder_64 = encoder_64.to(device)
head_64 = head_64.to(device)
print("Параметров головы 64:", sum(p.numel() for p in head_64.parameters()))

Параметров головы 64: 26663


In [11]:
trainer_64 = StatsHeadTrainer(
    encoder=encoder_64,
    head=head_64,
    train_loader=train_loader,
    eval_loader=eval_loader,
    output_dir=str(OUTPUT_DIR / "embed_64"),
    num_epochs=20,
    lr=1e-3,
    weight_decay=0.01,
    device=device,
    logging_steps=1,
    save_best=True,
)
trainer_64.train()

Epoch 1/20  Train MSE: 1022.214406  Eval MSE: 546.628088
Epoch 2/20  Train MSE: 449.395827  Eval MSE: 443.417975
Epoch 3/20  Train MSE: 404.067288  Eval MSE: 426.460095
Epoch 4/20  Train MSE: 387.923663  Eval MSE: 415.133255
Epoch 5/20  Train MSE: 379.317451  Eval MSE: 412.400388
Epoch 6/20  Train MSE: 374.024582  Eval MSE: 412.462610
Epoch 7/20  Train MSE: 370.734412  Eval MSE: 408.425092
Epoch 8/20  Train MSE: 365.932739  Eval MSE: 401.042761
Epoch 9/20  Train MSE: 366.153321  Eval MSE: 400.287979
Epoch 10/20  Train MSE: 361.412361  Eval MSE: 400.678635
Epoch 11/20  Train MSE: 359.106906  Eval MSE: 403.551437
Epoch 12/20  Train MSE: 356.811732  Eval MSE: 406.719962
Epoch 13/20  Train MSE: 354.895560  Eval MSE: 406.660223
Epoch 14/20  Train MSE: 354.711132  Eval MSE: 395.258981
Epoch 15/20  Train MSE: 352.296921  Eval MSE: 388.878918
Epoch 16/20  Train MSE: 351.171191  Eval MSE: 388.041110
Epoch 17/20  Train MSE: 348.717833  Eval MSE: 397.687604
Epoch 18/20  Train MSE: 349.133419  Eva

387.91254272460935

## 5. Evaluation

**Таблица сравнения (baseline vs embed_128 vs embed_64):**

- **Жирное** значение в строке — наименьший MSE среди baseline, embed_128 и embed_64 (лучший результат по этой статке).
- **%δ (128) vs (b)** и **%δ (64) vs (b)** — на сколько процентов MSE модели отличается от baseline (положительное = модель лучше, отрицательное = хуже).
  - **Синие** числа — модель лучше baseline (MSE ниже).
  - **Красные** числа — модель хуже baseline (MSE выше).
  - **0** — без изменения или baseline ≈ 0 (дельта не считается).

In [14]:
eval_mse, eval_mae = trainer.evaluate()
print(f"Eval MSE: {eval_mse:.6f}  |  Eval MAE: {eval_mae:.6f}")

Eval MSE: 237.248889  |  Eval MAE: 31.063934


In [17]:
from data.preprocessing import STAT_COLUMNS
import numpy as np
import pandas as pd
from baselines import compute_stats_baseline_mse_per_stat
from models.encoder import PlayerEncoder
from models.heads import StatsPredictionHead
from training.stats_trainer import StatsHeadTrainer

mse_ours = trainer.evaluate_per_stat(FORM_STATS_SIZE)
mse_baseline = compute_stats_baseline_mse_per_stat(eval_loader, FORM_STATS_SIZE, device=device)

# Модель embed_64 для таблицы
encoder_64 = PlayerEncoder(
    embed_size=64,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=FORM_STATS_SIZE,
    players_vocab_size=players_vocab_size,
    teams_vocab_size=teams_vocab_size,
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
state_64 = load_safetensors(ENCODER_CKPT_DIR_64 / "model.safetensors")
enc_state_64 = {k.replace("encoder.", ""): v for k, v in state_64.items() if k.startswith("encoder.")}
encoder_64.load_state_dict(enc_state_64, strict=True)
for p in encoder_64.parameters():
    p.requires_grad = False
head_64 = StatsPredictionHead(embed_size=64, num_stats=FORM_STATS_SIZE, hidden_dim=256)
if HEAD_PATH_64.exists():
    head_64.load_state_dict(torch.load(HEAD_PATH_64, map_location="cpu", weights_only=True))
encoder_64, head_64 = encoder_64.to(device), head_64.to(device)
trainer_64 = StatsHeadTrainer(encoder_64, head_64, train_loader, eval_loader, output_dir=str(OUTPUT_DIR), num_epochs=1, device=device)
mse_64 = trainer_64.evaluate_per_stat(FORM_STATS_SIZE)

# Если baseline ≈ 0, дельта не определена — считаем 0 (в т.ч. когда и там и там 0)
pct_128 = np.where(mse_baseline > 1e-9, (mse_baseline - mse_ours) / mse_baseline * 100, 0.0)
pct_64 = np.where(mse_baseline > 1e-9, (mse_baseline - mse_64) / mse_baseline * 100, 0.0)
pct_128_int = np.round(pct_128).astype(int)
pct_64_int = np.round(pct_64).astype(int)

best_col_arr = []
pct128_arr = []
pct64_arr = []
rows = []
for i in range(FORM_STATS_SIZE):
    vals = [mse_baseline[i], mse_ours[i], mse_64[i]]
    best_idx = int(np.argmin(vals))
    best_col_arr.append(best_idx)
    pct128_arr.append(pct_128_int[i])
    pct64_arr.append(pct_64_int[i])
    s128 = "+" if pct_128_int[i] > 0 else ("" if pct_128_int[i] == 0 else "")
    s64 = "+" if pct_64_int[i] > 0 else ("" if pct_64_int[i] == 0 else "")
    rows.append({
        "Statistics": STAT_COLUMNS[i],
        "baseline (b)": f"{mse_baseline[i]:.2f} | 0.00",
        "embed_128": f"{mse_ours[i]:.2f} | 0.00",
        "embed_64": f"{mse_64[i]:.2f} | 0.00",
        "%δ (128) vs (b)": f"{s128}{pct_128_int[i]}",
        "%δ (64) vs (b)": f"{s64}{pct_64_int[i]}",
    })

df_t = pd.DataFrame(rows)

def style_table(st):
    out = pd.DataFrame("", index=st.index, columns=st.columns)
    for i in range(len(st)):
        bc = best_col_arr[i]
        out.iloc[i, 1 + bc] = "font-weight: bold"
        if pct128_arr[i] > 0: out.iloc[i, 4] = "color: blue"
        elif pct128_arr[i] < 0: out.iloc[i, 4] = "color: red"
        if pct64_arr[i] > 0: out.iloc[i, 5] = "color: blue"
        elif pct64_arr[i] < 0: out.iloc[i, 5] = "color: red"
    return out

df_t.style.apply(style_table, axis=None)

C:\Users\vasilii\AppData\Local\Temp\ipykernel_4232\99926397.py:39: RuntimeWarning: divide by zero encountered in divide
  pct_128 = np.where(mse_baseline > 1e-9, (mse_baseline - mse_ours) / mse_baseline * 100, 0.0)
C:\Users\vasilii\AppData\Local\Temp\ipykernel_4232\99926397.py:40: RuntimeWarning: divide by zero encountered in divide
  pct_64 = np.where(mse_baseline > 1e-9, (mse_baseline - mse_64) / mse_baseline * 100, 0.0)


,Statistics,baseline (b),embed_128,embed_64,%δ (128) vs (b),%δ (64) vs (b)
0,pass_total,468.68 | 0.00,189.62 | 0.00,2300.33 | 0.00,+60,-391
1,pass_cross,2.32 | 0.00,1.93 | 0.00,5.49 | 0.00,+17,-137
2,pass_cut_back,0.08 | 0.00,0.08 | 0.00,0.87 | 0.00,+2,-937
3,pass_shot_assist,1.08 | 0.00,0.94 | 0.00,1.86 | 0.00,+13,-72
4,pass_goal_assist,0.06 | 0.00,0.07 | 0.00,3.30 | 0.00,-5,-5063
5,pass_no_touch,0.05 | 0.00,0.05 | 0.00,1.78 | 0.00,+1,-3406
6,pass_interception,0.00 | 0.00,0.00 | 0.00,3.01 | 0.00,0,0
7,pass_incomplete,18.60 | 0.00,13.58 | 0.00,65.88 | 0.00,+27,-254
8,pass_offside,0.00 | 0.00,0.00 | 0.00,1.78 | 0.00,0,0
9,pass_through_ball,0.23 | 0.00,0.24 | 0.00,1.32 | 0.00,-3,-481


In [ ]:
# При save_best=True голова уже сохранена в OUTPUT_DIR / stats_head.pt
print("Голова:", OUTPUT_DIR / "stats_head.pt")

Голова: C:\Users\vasilii\Documents\ML_project-football-\notebooks\stats_head_output\stats_head.pt
